# QC: Фин. рез. / АУР / Амортизация — Excel vs `final_df`

Самостоятельная сверка, без полного прогона `final_script_2`.

| Метрика | Lake | Excel |
|---------|------|-------|
| Фин. рез. | `SUM(fin_result)` | `Фин. Рез.` / `Fin.Res.` |
| АУР | `SUM(aur)` | `АУР` / `AUR` |
| Амортизация | `SUM(amortization)` | `Амортизация` |

По умолчанию Excel — **янв–июнь**. Июль: `header=-1`. Август подхватится, если файл есть.

Ячейка 5: по каждому месяцу **топ-5** договоров с наибольшим `|Фин.Рез. озеро − Excel|`. На пример — две строки (Excel / Озеро).


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')

excel_header = 0
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-07': -1,  # июль: строка заголовка не 0/1 — ищем по именам колонок
}

excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
    '2026-07': DATA_DIR / '07_Июль_2026.xlsx',
}

for extra_month, extra_name in [
    ('2026-07', '07_Июль_2026.xlsx'),
    ('2026-08', '08_Август_2026.xlsx'),
]:
    extra_path = DATA_DIR / extra_name
    if extra_path.exists():
        excel_reference_by_month[extra_month] = extra_path

final_df_candidates = [
    DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_08_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv',
]
checkpoint_dirs = [
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_08_final_script_2',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_07_final_script_2',
    DATA_DIR / 'checkpoints_final_df_2026_01_2026_06_mpos',
]

FIN_RESULT_EXCEL_COLS = [
    'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
    'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
    'fin_result', 'Fin.Res.', 'FinRes',
]
AUR_EXCEL_COLS = [
    'АУР', 'AUR', 'Aur', 'Аур', 'АУР, руб', 'АУР руб',
]
AMORT_EXCEL_COLS = [
    'Амортизация', 'Аморт', 'Амортизация терминалов',
    'amortization', 'Amortization', 'Амортизация, руб', 'Амортизация руб',
]
METRIC_SPECS = [
    ('fin_result', FIN_RESULT_EXCEL_COLS, 'Фин. рез.'),
    ('aur', AUR_EXCEL_COLS, 'АУР'),
    ('amortization', AMORT_EXCEL_COLS, 'Амортизация'),
]


def month_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if len(s) >= 7 and s[4] == '-':
        return s[:7]
    try:
        return pd.to_datetime(s).strftime('%Y-%m')
    except Exception:
        return None


def to_num_series(s):
    if s is None:
        return pd.Series(dtype='float64')
    if not isinstance(s, pd.Series):
        s = pd.Series(s)
    raw = s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False)
    raw = raw.str.replace(',', '.', regex=False)
    return pd.to_numeric(raw, errors='coerce')


def pick_col(columns, candidates):
    norm = {str(c).strip(): c for c in columns}
    for name in candidates:
        if name in norm:
            return norm[name]
    lower = {str(c).strip().lower().replace(' ', ''): c for c in columns}
    for name in candidates:
        key = name.strip().lower().replace(' ', '')
        if key in lower:
            return lower[key]
    return None


def read_month_excel(path, header):
    """header=-1: pandas так не умеет — ищем строку с АУР / Амортизация / Фин. рез."""
    if header != -1:
        return pd.read_excel(path, header=header)
    raw = pd.read_excel(path, header=None, nrows=12)
    markers = (
        'аур', 'амортизац', 'фин. рез', 'фин.рез', 'фин рез',
        'fin.res', 'fin_result',
    )
    for i, row in raw.iterrows():
        cells = ' '.join(
            str(x).lower().replace('\n', ' ')
            for x in row.tolist() if pd.notna(x)
        )
        if any(m in cells for m in markers):
            print(f'{path.name}: header=-1 → строка заголовка {i}')
            return pd.read_excel(path, header=i)
    print(f'{path.name}: header=-1, строка заголовка не найдена, header=None')
    return pd.read_excel(path, header=None)


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
for k, v in excel_reference_by_month.items():
    print(f'Excel {k}: exists={Path(v).exists()} | {v}')


In [ ]:
# 1) final_df: память → CSV → checkpoints
final_df = None
source_used = None

if 'final_df_period_df' in globals() and final_df_period_df is not None and len(final_df_period_df):
    final_df = final_df_period_df.copy()
    source_used = 'final_df_period_df (memory)'
elif 'final_df' in globals() and globals().get('final_df') is not None and len(globals()['final_df']):
    final_df = globals()['final_df'].copy()
    source_used = 'final_df (memory)'
else:
    for p in final_df_candidates:
        if p.exists():
            final_df = pd.read_csv(p, dtype=str, low_memory=False)
            source_used = f'csv: {p}'
            break

if final_df is None:
    frames = []
    used_dir = None
    for d in checkpoint_dirs:
        if not d.exists():
            continue
        month_files = sorted(d.glob('final_df_2026_*.parquet')) + sorted(d.glob('final_df_2026_*.csv'))
        if not month_files:
            continue
        used_dir = d
        seen = set()
        for f in month_files:
            key = f.stem
            if key in seen:
                continue
            seen.add(key)
            if f.suffix == '.parquet':
                frames.append(pd.read_parquet(f))
            else:
                frames.append(pd.read_csv(f, dtype=str, low_memory=False))
        break
    if frames:
        final_df = pd.concat(frames, ignore_index=True)
        source_used = f'checkpoints: {used_dir}'

if final_df is None or final_df.empty:
    raise RuntimeError(
        'final_df не найден. Положите CSV в DATA_DIR или откройте эту тетрадку '
        'после period-ячейки final_script_2.'
    )

need = {'report_month', 'fin_result', 'aur', 'amortization'}
missing = need - set(final_df.columns)
if missing:
    raise RuntimeError(f'В final_df нет колонок: {missing}. Есть: {list(final_df.columns)}')

work = final_df.copy()
work['report_month'] = work['report_month'].map(month_key)
for extra in ['fin_result', 'chod', 'aur', 'amortization']:
    if extra in work.columns:
        work[extra] = to_num_series(work[extra])

print('source:', source_used)
print('rows:', f'{len(work):,}', '| months:', sorted(work['report_month'].dropna().unique().tolist()))
display(work.groupby('report_month', as_index=False)[['fin_result', 'aur', 'amortization']].sum())


In [ ]:
# 2) Excel: SUM по Фин. рез. / АУР / Амортизация
excel_rows = []
for month, path in excel_reference_by_month.items():
    path = Path(path)
    row = {
        'report_month': month,
        'excel_rows': 0,
        'excel_path': str(path),
        'file_status': 'ok',
    }
    for key, _cands, _label in METRIC_SPECS:
        row[f'{key}_excel'] = np.nan
        row[f'{key}_excel_col'] = None
        row[f'{key}_status'] = 'file_missing'
    if not path.exists():
        row['file_status'] = 'file_missing'
        excel_rows.append(row)
        continue
    header = excel_header_by_month.get(month, excel_header)
    ex = read_month_excel(path, header)
    row['excel_rows'] = int(len(ex))
    for key, cands, label in METRIC_SPECS:
        col = pick_col(ex.columns, cands)
        if col is None:
            row[f'{key}_status'] = 'column_missing'
            print(f'{month}: нет колонки {label}. Колонки: {list(ex.columns)}')
            continue
        row[f'{key}_excel'] = float(to_num_series(ex[col]).fillna(0).sum())
        row[f'{key}_excel_col'] = col
        row[f'{key}_status'] = 'has_excel_reference'
    excel_rows.append(row)

excel_month = pd.DataFrame(excel_rows)
print('Excel months (колонки, которые нашлись):')
display(excel_month[[
    'report_month', 'file_status', 'excel_rows',
    'fin_result_excel', 'fin_result_excel_col', 'fin_result_status',
    'aur_excel', 'aur_excel_col', 'aur_status',
    'amortization_excel', 'amortization_excel_col', 'amortization_status',
]])


In [ ]:
# 3) Сводка lake vs Excel по трём метрикам
agg = {c: 'sum' for c in ['fin_result', 'chod', 'aur', 'amortization'] if c in work.columns}
lake_month = work.groupby('report_month', as_index=False).agg(agg)
lake_month = lake_month.rename(columns={
    'fin_result': 'fin_result_lake',
    'chod': 'chod_lake',
    'aur': 'aur_lake',
    'amortization': 'amortization_lake',
})

cmp = lake_month.merge(excel_month, on='report_month', how='outer')
cmp = cmp.sort_values('report_month').reset_index(drop=True)

for key, _cands, _label in METRIC_SPECS:
    lake_c = f'{key}_lake'
    excel_c = f'{key}_excel'
    cmp[f'{key}_delta'] = cmp[lake_c] - cmp[excel_c]
    cmp[f'{key}_delta_pct'] = np.where(
        cmp[excel_c].abs() > 1e-9,
        100.0 * cmp[f'{key}_delta'] / cmp[excel_c],
        np.nan,
    )
    # Excel часто хранит расход со знаком «−»; lake aur/amort — положительные
    cmp[f'{key}_delta_abs'] = cmp[lake_c].abs() - cmp[excel_c].abs()

if {'chod_lake', 'aur_lake', 'amortization_lake', 'fin_result_lake'}.issubset(cmp.columns):
    cmp['fin_result_from_parts'] = (
        cmp['chod_lake'].fillna(0) - cmp['aur_lake'].fillna(0) - cmp['amortization_lake'].fillna(0)
    )
    cmp['parts_minus_fin_result'] = cmp['fin_result_from_parts'] - cmp['fin_result_lake'].fillna(0)


def _print_metric(key, title):
    status_c = f'{key}_status'
    lake_c = f'{key}_lake'
    excel_c = f'{key}_excel'
    cols = [
        'report_month', lake_c, excel_c,
        f'{key}_delta', f'{key}_delta_pct', f'{key}_delta_abs',
        f'{key}_excel_col', status_c,
    ]
    print(f'=== {title}: lake vs Excel ===')
    display(cmp[cols])
    with_xl = cmp.loc[cmp[status_c] == 'has_excel_reference']
    sum_lake_xl = float(with_xl[lake_c].fillna(0).sum()) if len(with_xl) else 0.0
    sum_excel = float(with_xl[excel_c].fillna(0).sum()) if len(with_xl) else 0.0
    sum_delta = sum_lake_xl - sum_excel
    sum_delta_pct = (sum_delta / sum_excel * 100.0) if abs(sum_excel) > 1e-9 else np.nan
    sum_delta_abs = (
        float(with_xl[lake_c].abs().fillna(0).sum()) - float(with_xl[excel_c].abs().fillna(0).sum())
        if len(with_xl) else np.nan
    )
    print(f'=== Итого {title} (месяцы, где колонка нашлась) ===')
    print(f'  lake  = {sum_lake_xl:,.2f}')
    print(f'  excel = {sum_excel:,.2f}')
    print(f'  delta (lake-excel) = {sum_delta:,.2f}')
    print(f'  delta_pct = {sum_delta_pct:,.2f}%' if pd.notna(sum_delta_pct) else '  delta_pct = n/a')
    print(f'  delta по модулю (|lake|-|excel|) = {sum_delta_abs:,.2f}')
    print(f'  lake все месяцы = {float(cmp[lake_c].fillna(0).sum()):,.2f}')
    print()


_print_metric('fin_result', 'Фин. рез.')
_print_metric('aur', 'АУР')
_print_metric('amortization', 'Амортизация')

if {'chod_lake', 'aur_lake', 'amortization_lake'}.issubset(cmp.columns):
    print('=== Компоненты lake: chod − aur − amort ≈ fin_result ===')
    display(cmp[[
        'report_month', 'fin_result_lake', 'fin_result_from_parts', 'parts_minus_fin_result',
        'chod_lake', 'aur_lake', 'amortization_lake',
    ]])

out_csv = DATA_DIR / 'fin_result_aur_amort_lake_vs_excel_by_month.csv'
cmp.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('Saved:', out_csv)


In [ ]:
# 4) Топ-5 расхождений Фин. рез. по каждому месяцу (Excel vs озеро)
# На каждый пример — две строки: Excel и Озеро.

INN_COLS = ['ИНН', 'inn', 'c_inn']
AGR_COLS = ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id']
NAME_COLS = [
    'Наименование', 'Название', 'company_name', 'Наименование клиента',
    'Наименование ТСП', 'Клиент', 'Организация',
]
RETL_COLS = ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек', 'retl_cnt']
TRX_CNT_COLS = ['Количество операций', 'Количеств операций', 'Количество транзакций', 'trx_cnt']
TRX_SUM_COLS = ['Сумма операций', 'Сумма опреаций', 'сумма транзакций', 'trx_sum']
CHOD_COLS = ['ЧОД', 'chod']

OUT_COLS = [
    'месяц', 'ранг', 'источник', 'delta_финрез',
    'ИНН', 'Наименование', 'agr_id',
    'Количество торговых точек', 'Количество транзакций', 'сумма транзакций',
    'АУР', 'Амортизация', 'ЧОД', 'Фин.Рез.',
]


def _first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else None


def _excel_month_agr(path, header):
    ex = read_month_excel(path, header)
    resolved = {
        'inn': pick_col(ex.columns, INN_COLS),
        'agr': pick_col(ex.columns, AGR_COLS),
        'name': pick_col(ex.columns, NAME_COLS),
        'retl': pick_col(ex.columns, RETL_COLS),
        'trx_cnt': pick_col(ex.columns, TRX_CNT_COLS),
        'trx_sum': pick_col(ex.columns, TRX_SUM_COLS),
        'aur': pick_col(ex.columns, AUR_EXCEL_COLS),
        'amort': pick_col(ex.columns, AMORT_EXCEL_COLS),
        'chod': pick_col(ex.columns, CHOD_COLS),
        'fin': pick_col(ex.columns, FIN_RESULT_EXCEL_COLS),
    }
    if resolved['inn'] is None or resolved['agr'] is None or resolved['fin'] is None:
        return None, resolved
    t = pd.DataFrame({
        'inn_key': ex[resolved['inn']].map(normalize_inn_q1),
        'agr_id_key': ex[resolved['agr']].map(normalize_agr_q1),
        'Наименование': ex[resolved['name']] if resolved['name'] else None,
        'Количество торговых точек': to_num_series(ex[resolved['retl']]) if resolved['retl'] else np.nan,
        'Количество транзакций': to_num_series(ex[resolved['trx_cnt']]) if resolved['trx_cnt'] else np.nan,
        'сумма транзакций': to_num_series(ex[resolved['trx_sum']]) if resolved['trx_sum'] else np.nan,
        'АУР': to_num_series(ex[resolved['aur']]) if resolved['aur'] else np.nan,
        'Амортизация': to_num_series(ex[resolved['amort']]) if resolved['amort'] else np.nan,
        'ЧОД': to_num_series(ex[resolved['chod']]) if resolved['chod'] else np.nan,
        'Фин.Рез.': to_num_series(ex[resolved['fin']]),
    })
    t = t.dropna(subset=['inn_key', 'agr_id_key'])
    agg = t.groupby(['inn_key', 'agr_id_key'], as_index=False).agg({
        'Наименование': _first_nonnull,
        'Количество торговых точек': 'max',
        'Количество транзакций': 'sum',
        'сумма транзакций': 'sum',
        'АУР': 'sum',
        'Амортизация': 'sum',
        'ЧОД': 'sum',
        'Фин.Рез.': 'sum',
    })
    return agg, resolved


def _lake_month_agr(month_df):
    t = month_df.copy()
    t['inn_key'] = t['inn'].map(normalize_inn_q1) if 'inn' in t.columns else None
    t['agr_id_key'] = t['agr_id'].map(normalize_agr_q1)
    name_col = 'company_name' if 'company_name' in t.columns else None
    t['Наименование'] = t[name_col] if name_col else None
    t['Количество торговых точек'] = to_num_series(t['retl_cnt']) if 'retl_cnt' in t.columns else np.nan
    t['Количество транзакций'] = to_num_series(t['trx_cnt']) if 'trx_cnt' in t.columns else np.nan
    t['сумма транзакций'] = to_num_series(t['trx_sum']) if 'trx_sum' in t.columns else np.nan
    t['АУР'] = to_num_series(t['aur']) if 'aur' in t.columns else np.nan
    t['Амортизация'] = to_num_series(t['amortization']) if 'amortization' in t.columns else np.nan
    t['ЧОД'] = to_num_series(t['chod']) if 'chod' in t.columns else np.nan
    t['Фин.Рез.'] = to_num_series(t['fin_result'])
    t = t.dropna(subset=['inn_key', 'agr_id_key'])
    return t.groupby(['inn_key', 'agr_id_key'], as_index=False).agg({
        'Наименование': _first_nonnull,
        'Количество торговых точек': 'max',
        'Количество транзакций': 'sum',
        'сумма транзакций': 'sum',
        'АУР': 'sum',
        'Амортизация': 'sum',
        'ЧОД': 'sum',
        'Фин.Рез.': 'sum',
    })


detail_rows = []
for month, path in excel_reference_by_month.items():
    path = Path(path)
    if not path.exists():
        print(f'{month}: Excel нет, skip')
        continue
    header = excel_header_by_month.get(month, excel_header)
    ex_agr, resolved = _excel_month_agr(path, header)
    if ex_agr is None:
        print(f'{month}: не собрались ключи/Фин.Рез. resolved={resolved}')
        continue
    lake_m = work.loc[work['report_month'] == month]
    if lake_m.empty:
        print(f'{month}: в final_df нет строк')
        continue
    lk_agr = _lake_month_agr(lake_m)
    joined = ex_agr.merge(lk_agr, on=['inn_key', 'agr_id_key'], how='outer', suffixes=('_excel', '_lake'))
    joined['delta_финрез'] = joined['Фин.Рез._lake'].fillna(0) - joined['Фин.Рез._excel'].fillna(0)
    joined['abs_delta'] = joined['delta_финрез'].abs()
    top = joined.sort_values('abs_delta', ascending=False).head(5).reset_index(drop=True)
    print(f'\n===== {month}: топ-5 |delta Фин.Рез.| =====')
    for i, r in top.iterrows():
        rank = i + 1
        name = r.get('Наименование_lake') or r.get('Наименование_excel')
        common = {
            'месяц': month,
            'ранг': rank,
            'delta_финрез': r['delta_финрез'],
            'ИНН': r['inn_key'],
            'Наименование': name,
            'agr_id': r['agr_id_key'],
        }
        excel_row = {
            **common,
            'источник': 'Excel',
            'Количество торговых точек': r.get('Количество торговых точек_excel'),
            'Количество транзакций': r.get('Количество транзакций_excel'),
            'сумма транзакций': r.get('сумма транзакций_excel'),
            'АУР': r.get('АУР_excel'),
            'Амортизация': r.get('Амортизация_excel'),
            'ЧОД': r.get('ЧОД_excel'),
            'Фин.Рез.': r.get('Фин.Рез._excel'),
        }
        lake_row = {
            **common,
            'источник': 'Озеро',
            'Количество торговых точек': r.get('Количество торговых точек_lake'),
            'Количество транзакций': r.get('Количество транзакций_lake'),
            'сумма транзакций': r.get('сумма транзакций_lake'),
            'АУР': r.get('АУР_lake'),
            'Амортизация': r.get('Амортизация_lake'),
            'ЧОД': r.get('ЧОД_lake'),
            'Фин.Рез.': r.get('Фин.Рез._lake'),
        }
        detail_rows.extend([excel_row, lake_row])

top5_detail = pd.DataFrame(detail_rows, columns=OUT_COLS)
print('\n=== Все примеры: Excel и Озеро отдельными строками ===')
display(top5_detail)

out_top5 = DATA_DIR / 'fin_result_top5_delta_excel_vs_lake_by_month.csv'
top5_detail.to_csv(out_top5, index=False, encoding='utf-8-sig')
print('Saved:', out_top5)
